In [0]:
%sql
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
LIMIT 10;

#PARTE 1: Limpieza, Exploración y Modificaciones (10 Ejercicios Básicos/Intermedios)

### 1.(CREATE / INSERT) El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_reservas_prueba
USING DELTA
AS
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE 1 = 0;

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_reservas_prueba
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE hotel = 'City Hotel' AND is_canceled = 1;

In [0]:
%sql
SELECT
    hotel,
    is_canceled,
    COUNT(*) AS total_reservas
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
GROUP BY hotel, is_canceled;

Como se puede ver hubieron 33102 reservas que fueron canceladas del hotel "City Hotel" 

las cuales fueron modificadas en una tabla de prueba para no afectar a la tabla original.

###2. (UPDATE) Se detectó un error en los sistemas de captura de datos: los huéspedes que registraron babies en realidad no tenían bebés, sino que era un error del canal. Actualiza la tabla tb_reservas_prueba y cambia el valor de babies a 0 para todos los registros.

In [0]:
%sql
UPDATE cursodataacademy.proyecto_hotel.tb_reservas_prueba
SET babies = 0;

In [0]:
%sql
SELECT
    babies,
    COUNT(*) AS total_reservas
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
GROUP BY babies;

Se pudo corregir el error de las personas que registraron que tenian bebes, lo cual no 

es asi, se procedio a colocar 0 en vez de 1 en todos los registros como se puede ver se aplico en 33102 filas

### 3. (CAST / CONCAT) Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").

In [0]:
%sql
SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT(
        CAST(arrival_date_day_of_month AS STRING),
        '-',
        arrival_date_month,
        '-',
        CAST(arrival_date_year AS STRING)
    ) AS arrival_date_string
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Se junto las columnas day, month y year en una sola "un formato mas legible", en 

este caso se volvio string day y year, ya que month ya estaba en formato string

###4. (LIKE) Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.

In [0]:
%sql
SELECT
    distribution_channel
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel LIKE 'T%';

In [0]:
%sql
SELECT
    distribution_channel,
    COUNT(*) AS TOTAL_DISTRIBUTION
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel LIKE 'T%'
GROUP BY distribution_channel;

Aqui lo que se hizo fue filtrar utilizando like para que de la columna distribution 

solo aparezcan la que comienzan por la letra "T" y en la siguiente consulta es mas 

que nada para saber con exactitud cuantas filas son

###5. (COUNT / SUM / AVG) Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas,
    SUM(total_of_special_requests) AS total_solicitudes_especiales,
    ROUND(AVG(adr),2) AS precio_promedio_noche
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Tenemos 119390 reservas y 68215 solicitudes especiales, con un promedio de 

USD 101.83 por noche. 

###6. (DELETE) En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas_restantes
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba;

In [0]:
%sql
DELETE FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
WHERE adr = 0.0;

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas_restantes
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba;

En la primera consulta quise ver cuantas filas tenia en total 

luego procedí a eliminar los que tenian promedio 0.0, y se 

eliminaron 129, luego de volver a colocar la primera consulta se vio una disminucion de 33102 a 32973

###7. (GROUP BY) El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.

In [0]:
%sql
SELECT
    distribution_channel,
    COUNT(*) AS total_reservas
FROM cursodataacademy.proyecto_hotel.hotel_bookings
GROUP BY distribution_channel
ORDER BY total_reservas DESC;

Podemos ver el resultado de esta consulta que TA/TO tiene la mayor 

cantidad de reservas y que hay 5 que no estan definidas de que canal vienen. 

###8. (CASE WHEN / Tipos de Datos) Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium".

In [0]:
%sql
SELECT
    hotel,
    adr,
    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        WHEN adr > 150 THEN 'Premium'
    END AS categoria_precio
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Gracias a esta consulta podemos tener un tipo de clase por el adr de las reservas

###9. (Funciones de Fecha - DATE_SUB / DATE_ADD) (Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio). Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.

In [0]:
%sql
ALTER TABLE cursodataacademy.proyecto_hotel.hotel_bookings
ADD COLUMNS (arrival_date DATE);

In [0]:
%sql
UPDATE cursodataacademy.proyecto_hotel.hotel_bookings
SET arrival_date = TO_DATE(
    CONCAT(
        arrival_date_day_of_month, '-',
        arrival_date_month, '-',
        arrival_date_year
    ),
    'd-MMMM-yyyy'
);

In [0]:
%sql
SELECT
    arrival_date,
    lead_time,
    DATE_SUB(arrival_date, CAST(lead_time AS INT)) AS booking_date
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Aqui se agrego una columna para poder tener la fecha de llegada toda 

junta ya que estaba separada por dia, mes y año, entonces lo restamos 

con la columna que tenia los dias de anticipacion y con ello tuvimos la fecha exacta que hicieron la reservación

###10. (Funciones Matemáticas) Queremos calcular una "cuota de riesgo". Selecciona el lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada meses_anticipacion_techo.

In [0]:
%sql
SELECT
    lead_time,
    CEIL(lead_time / 30.0) AS meses_anticipacion_techo
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Gracias a esta consulta podemos ver los meses aprox con la que la 

gente anticipa su reserva y con la función ceil pudimos redondear este numero. 

#PARTE 2: Impacto Financiero y JOINS (6 Ejercicios de Negocio)

###11. El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas_canceladas,
    COUNT(*) * 1.5 AS perdida_total_usd
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel = 'TA/TO' AND is_canceled = 1;

Con esta consulta podemos ver que del canal TA/TO se multiplico el 1.5 

USD por las reservas que fueron canceladas y en total se le pagó a la agencia 60228

###12. El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
SELECT
    COUNT(*) AS total_cancelaciones_tardias,
    COUNT(*) * 120 AS perdida_total_usd
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE is_canceled = 1 AND DATEDIFF(arrival_date, TO_DATE(reservation_status_date, 'd/MM/yy')) BETWEEN 0 AND 2;

en esta consulta podemos ver que en cancelaciones de reversas menores 

a 3 dias el hotel pierde 120 USD por lo tanto hay un total de 3204 reservas de este tipo y la perdida total es de 384480

###13. (JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_hotel.tb_paises (
    cod_pais STRING,
    nombre_pais STRING
);

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_paises
(cod_pais, nombre_pais)
VALUES
    ('PRT', 'Portugal'),
    ('ESP', 'España'),
    ('FRA', 'Francia'),
    ('USA', 'Estados Unidos');

In [0]:
%sql
SELECT
    r.is_canceled,
    p.nombre_pais
FROM cursodataacademy.proyecto_hotel.hotel_bookings AS r
LEFT JOIN cursodataacademy.proyecto_hotel.tb_paises AS p
    ON r.country = p.cod_pais;

al no tener la tabla procedi a crearla tb_paises y agregue unos paises 

con su cod 3 letras que representen o se de a entender que es ese pais 

y como podemos ver se hace una union entre estas dos tablas gracias al left join

###14. (JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?

In [0]:
%sql
SELECT
    p.nombre_pais,
    COUNT(*) AS total_reservas_canceladas
FROM cursodataacademy.proyecto_hotel.hotel_bookings AS r
INNER JOIN cursodataacademy.proyecto_hotel.tb_paises AS p
    ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY total_reservas_canceladas DESC
LIMIT 5;

Gracias a esta consulta podemos ver la cantidad de reservas canceladas 

por paises y como podemos ver portugal tiene la mayor cantidad con 

27519 y al solo colocar 4 paises en la anterior al crear la tabla solo aparecen 4 y no 5

###15. (JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_hotel.tb_vendedores (
    id_vendedor INT,
    nombre_vendedor STRING);

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_vendedores
(id_vendedor, nombre_vendedor)
VALUES
    (1, 'Carlos Pérez'),
    (2, 'María López'),
    (3, 'Juan Rodríguez'),
    (4, 'Ana Torres');

In [0]:
%sql
ALTER TABLE cursodataacademy.proyecto_hotel.hotel_bookings
ADD COLUMNS (id_vendedor INT);

In [0]:
%sql
UPDATE cursodataacademy.proyecto_hotel.hotel_bookings
SET id_vendedor = CASE
    WHEN distribution_channel = 'Direct' THEN 1
    WHEN distribution_channel = 'Corporate' THEN 2
    WHEN distribution_channel = 'TA/TO' THEN 3
    ELSE NULL
END;

In [0]:
%sql
SELECT
    r.distribution_channel,
    v.nombre_vendedor
FROM cursodataacademy.proyecto_hotel.hotel_bookings AS r
FULL OUTER JOIN cursodataacademy.proyecto_hotel.tb_vendedores AS v
    ON r.id_vendedor = v.id_vendedor;

###16. (JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).


In [0]:
%sql
ALTER TABLE cursodataacademy.proyecto_hotel.hotel_bookings
ADD COLUMNS (id_reserva INT);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vista_reservas_ids AS
SELECT
    CAST(ROW_NUMBER() OVER (
        ORDER BY hotel, arrival_date, lead_time
    ) AS INT) AS id_reserva,
    hotel
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cursodataacademy.proyecto_hotel.tb_facturas (
    id_reserva INT,
    monto_facturado DOUBLE
);

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_facturas
VALUES
    (1, 250.00),
    (2, 400.00),
    (3, 180.00),
    (200000, 350.00);

In [0]:
%sql
SELECT
    r.hotel,
    f.id_reserva,
    f.monto_facturado
FROM vista_reservas_ids AS r
RIGHT JOIN cursodataacademy.proyecto_hotel.tb_facturas AS f
    ON r.id_reserva = f.id_reserva;

#PARTE 3: Validación de Hipótesis de Cancelación (4 Ejercicios Analíticos)

###17. Hipótesis 1: ¿Los hijos aseguran el viaje? Escribe una consulta usando CASE WHEN que clasifique a la reserva como "Con Hijos" (si children > 0 o babies > 0) o "Sin Hijos". Agrupa por esta nueva categoría y calcula la tasa de cancelación (puedes sacar el promedio - AVG - de la columna is_canceled). ¿Qué segmento cancela más?

In [0]:
%sql
SELECT
    CASE
        WHEN children > 0 OR babies > 0 THEN 'Con Hijos'
        ELSE 'Sin Hijos'
    END AS categoria,
    COUNT(*) AS total_reservas,
    ROUND(AVG(is_canceled) * 100, 2) AS tasa_cancelacion
FROM cursodataacademy.proyecto_hotel.hotel_bookings
GROUP BY
    CASE
        WHEN children > 0 OR babies > 0 THEN 'Con Hijos'
        ELSE 'Sin Hijos'
    END;

Con esta consulta podemos ver que las reservas sin hijos son las que mayor cancelación, 

se podría decir que viajar con hijos tiene menos probabilidad de cancelar la reserva.

###18. Hipótesis 2: El impacto de las solicitudes especiales. Agrupa por la columna total_of_special_requests y calcula la tasa de cancelación (AVG(is_canceled)) para cada cantidad de solicitudes (0, 1, 2, etc.). Ordena por la cantidad de solicitudes de forma ascendente. ¿A medida que el cliente pide más cosas, cancela más o cancela menos?

In [0]:
%sql
SELECT
    total_of_special_requests AS cantidad_solicitudes,
    COUNT(*) AS total_reservas,
    ROUND(AVG(is_canceled) * 100, 2) AS tasa_cancelacion
FROM cursodataacademy.proyecto_hotel.hotel_bookings
GROUP BY total_of_special_requests
ORDER BY total_of_special_requests ASC;

Gracias a esta consulta podemos decir que las reservas sin solicitudes tienen un 47.72% de cancelación 

y las que tienen 5 solicitudes tienen un 5%. 

Así que los clientes que suelen hacer mas solicitudes se podría decir que son los que tienen un mayor compromiso con su reserva. 

###19. Hipótesis 3: ¿Cambiar la reserva es señal de compromiso? Al igual que en los ejercicios anteriores, utiliza CASE WHEN para segmentar la columna booking_changes en tres grupos: "Sin Cambios" (0), "1 a 3 cambios" y "Más de 3 cambios". Calcula la tasa de cancelación para cada grupo. ¿Quiénes tienen menor riesgo de cancelar?

In [0]:
%sql
SELECT
    CASE
        WHEN booking_changes = 0 THEN 'Sin Cambios'
        WHEN booking_changes BETWEEN 1 AND 3 THEN '1 a 3 cambios'
        ELSE 'Más de 3 cambios'
    END AS categoria,
    COUNT(*) AS total_reservas,
    ROUND(AVG(is_canceled) * 100, 2) AS tasa_cancelacion
FROM cursodataacademy.proyecto_hotel.hotel_bookings
GROUP BY
    CASE
        WHEN booking_changes = 0 THEN 'Sin Cambios'
        WHEN booking_changes BETWEEN 1 AND 3 THEN '1 a 3 cambios'
        ELSE 'Más de 3 cambios'
    END;

Con esta consulta podemos decir que las reservas que tienen 1 a 3 

cambios tienen una tasa de cancelación del 15.59% mientras que las que no tienen cambios tienen un 40.85% 

###20. Hipótesis 4: El factor precio (adr). Agrupa las reservas por tipo de hotel (hotel). Para cada tipo, muestra la cantidad total de cancelaciones y el precio promedio (adr) solo de las reservas canceladas. En base a tu consulta, ¿en qué tipo de hotel el precio promedio de cancelación es más alto?

In [0]:
%sql
SELECT
    hotel,
    COUNT(*) AS total_cancelaciones,
    ROUND(AVG(adr), 2) AS precio_promedio_cancelacion
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE is_canceled = 1
GROUP BY hotel
ORDER BY precio_promedio_cancelacion DESC;

Con esta consulta podemos decir que el resort hotel tiene un mayor 

precio promedio entre las reservas canceladas 105.79, luego el city 

hotel que cuenta con 104.69 pero city hotel cuenta con mayor cantidad de cancelaciones 33102 respecto a resort hotel con 11122